### merging all the ptrb baselines

In [ ]:
import json
import glob
import os

# Directory containing your files
dir_path = "/home/ojas/scripts/datasets/Final_Runs"

# Output file
output_file = os.path.join(dir_path, "All_Reasoning_PTRB_BASELINES.json")

# Pattern for matching your baseline JSONs
pattern = os.path.join(dir_path, "*PTRB_BASELINES.json")

# Find all matching files
file_paths = glob.glob(pattern)

print("Found files:")
for f in file_paths:
    print(" -", f)

merged_data = []

# Read and merge all files
for file_path in file_paths:
    print(f"\nProcessing: {os.path.basename(file_path)}")
    with open(file_path, "r") as fp:
        data = json.load(fp)

        # If each JSON is a list
        if isinstance(data, list):
            merged_data.extend(data)
        else:
            # If it's a dict with a key like "data" or something else
            print(" Warning: JSON is not a list. Attempting to extract list values...")
            for v in data.values():
                if isinstance(v, list):
                    merged_data.extend(v)

print(f"\nMerging completed. Total entries: {len(merged_data)}")

# Write output file
with open(output_file, "w") as fp:
    json.dump(merged_data, fp, indent=2)

print(f"\nSaved merged JSON → {output_file}")

### collecting dataset and shots and models reasoning sets for ptrb baselines

In [ ]:
import json
import os

# ==========================
# CONFIG: CHANGE ONLY THESE
# ==========================
DATASET_FILTER = "LIAR"
SHOTS_FILTER = 1
MODEL_FILTER = None      # or set e.g. "Deepseek_Qwen_14B"

# Paths
DIR = "/home/ojas/scripts/datasets/Final_Runs"
MERGED_FILE = os.path.join(DIR, "All_Reasoning_PTRB_BASELINES.json")

# Output file name auto-generated
outname_parts = [p for p in [
    DATASET_FILTER,
    f"shots{SHOTS_FILTER}",
    MODEL_FILTER
] if p is not None]

OUTPUT_FILE = os.path.join(
    DIR,
    "filtered_" + "_".join(outname_parts) + ".json"
)

# ==========================
# LOAD MERGED JSON
# ==========================
print(f"Loading merged file: {MERGED_FILE}")
with open(MERGED_FILE, "r") as fp:
    data = json.load(fp)

print(f"Total entries loaded: {len(data)}")

# ==========================
# FILTER LOGIC
# ==========================
filtered = []

for entry in data:
    if entry.get("dataset") != DATASET_FILTER:
        continue

    if entry.get("shots") != SHOTS_FILTER:
        continue

    if MODEL_FILTER is not None:
        if entry.get("model") != MODEL_FILTER:
            continue

    filtered.append(entry)

print(f"Filtered entries: {len(filtered)}")

# ==========================
# SAVE OUTPUT
# ==========================
with open(OUTPUT_FILE, "w") as fp:
    json.dump(filtered, fp, indent=2)

print(f"Saved filtered subset → {OUTPUT_FILE}")

### for SHOTS_ANALYSIS Table (no perturbation)

In [ ]:
# collecting files

import json
import re
from pathlib import Path

# Directory containing your 15 files
DATA_DIR = Path(".")

# Collect files automatically (optional)
files = list(DATA_DIR.glob("*shot_BASELINES.json"))

print(f"Found {len(files)} input files.")
if len(files) == 0:
    raise RuntimeError("No *_shot_BASELINES.json files found.")

# Output containers
merged = {
    1: [],
    2: [],
    4: []
}

# Regex to extract shot count, e.g. "2shot" → 2
shot_pattern = re.compile(r"(\d)shot", re.IGNORECASE)

for file in files:
    fname = file.name
    match = shot_pattern.search(fname)
    if not match:
        print(f"❌ Could not detect shot count in: {fname}")
        continue

    shot = int(match.group(1))
    if shot not in merged:
        print(f"⚠️ Shot count {shot} in {fname} not recognized (expected 1/2/4)")
        continue

    print(f"📄 Reading file for {shot}-shot: {fname}")

    # Load JSON list entries
    with file.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        print(f"⚠️ File {fname} does not contain a list — skipping.")
        continue

    merged[shot].extend(data)
    print(f"   ➜ Added {len(data)} entries.")

# Output final merged files
output_files = {
    1: "RT_Original_1shot.json",
    2: "RT_Original_2shot.json",
    4: "RT_Original_4shot.json"
}

for shot, outname in output_files.items():
    with open(outname, "w", encoding="utf-8") as f:
        json.dump(merged[shot], f, indent=2, ensure_ascii=False)
    print(f"✅ Saved {len(merged[shot])} entries → {outname}")

### human eval datasets

import json
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------
# File → Dataset column
# ---------------------
file_to_dataset = {
    # "TESTBASELINES.json":"test",
    "entbank_random_150_R4-10_E_BASELINES_new.json": "ENTAILMENT_BANK",
    "proof_random_150_R4-10_E_BASELINES_new.json": "PROOFWRITER",
    "math_random_150_R4-R10_BASELINES.json": "GSM",
    "strat_random_150_R3+_E_BASELINES_new.json": "STRATQA",
    "pubhealth_random_150_R4-10_BASELINES.json": "PUBMED",
}

files = list(file_to_dataset.keys())

# Metrics (rows in final CSV)
metric_rows = [
    "ROSCOE-SA",
    "ROSCOE-SS",
    "ROSCOE-LI",
    "ROSCOE-LC",
    "ROSCOE_MEAN",
    "LLM_AS_A_JUDGE",
    "KOTONYA_AND_TONI",
    "RECEval",
    "OM_COHERENCE (B1)",
    "OM_QUALITY (B2)",
    "OM_EVIDENCE (B3)",
    "OM_B1_B2",
    "OM_B2_B3",
    "OM_B1_B3",
    "OM_MEAN"
]

# A dict to collect column outputs
summary = {metric: {} for metric in metric_rows}

# Utility to safely get nested values
def get_float(entry, *keys):
    cur = entry
    try:
        for k in keys:
            cur = cur[k]
        return float(cur)
    except:
        return np.nan

for filename in files:
    dataset_name = file_to_dataset[filename]
    path = Path(filename)

    if not path.exists():
        print(f"⚠️ Missing file: {filename}")
        for m in metric_rows:
            summary[m][dataset_name] = np.nan
        continue

    with path.open("r") as f:
        data = json.load(f)

    # Collect lists
    sa = []
    ss = []
    li = []
    lc = []
    judge = []
    coherence = []
    receval = []

    om_coh = []
    om_quality = []
    om_evidence = []
    om_b1_b2 = []
    om_b2_b3 = []
    om_b3_b1 = []
    om_total = []

    for entry in data:
        sa.append(get_float(entry, "mean_ROSCOE_SA"))
        ss.append(get_float(entry, "mean_ROSCOE_SS"))
        li.append(get_float(entry, "mean_ROSCOE_LI"))
        lc.append(get_float(entry, "mean_ROSCOE_LC"))
        judge.append(get_float(entry, "judge_score"))
        coherence.append(get_float(entry, "coherence_scores", "mean_coherence"))
        receval.append(get_float(entry, "mean_RECEval"))

        om_coh.append(get_float(entry, "ourmetric", "coherence_score"))
        om_quality.append(get_float(entry, "ourmetric", "quality_score"))
        om_evidence.append(get_float(entry, "ourmetric", "evidence_score"))
        om_b1_b2.append(get_float(entry, "ourmetric", "b1_b2"))
        om_b2_b3.append(get_float(entry, "ourmetric", "b2_b3"))
        om_b3_b1.append(get_float(entry, "ourmetric", "b3_b1"))
        om_total.append(get_float(entry, "ourmetric", "total_score"))

    # Compute means
    mean_sa = np.nanmean(sa)
    mean_ss = np.nanmean(ss)
    mean_li = np.nanmean(li)
    mean_lc = np.nanmean(lc)

    roscoe_mean = np.nanmean([mean_sa, mean_ss, mean_li, mean_lc])

    summary["ROSCOE-SA"][dataset_name] = mean_sa
    summary["ROSCOE-SS"][dataset_name] = mean_ss
    summary["ROSCOE-LI"][dataset_name] = mean_li
    summary["ROSCOE-LC"][dataset_name] = mean_lc
    summary["ROSCOE_MEAN"][dataset_name] = roscoe_mean

    summary["LLM_AS_A_JUDGE"][dataset_name] = np.nanmean(judge)
    summary["KOTONYA_AND_TONI"][dataset_name] = np.nanmean(coherence)
    summary["RECEval"][dataset_name] = np.nanmean(receval)

    summary["OM_COHERENCE (B1)"][dataset_name] = np.nanmean(om_coh)
    summary["OM_QUALITY (B2)"][dataset_name] = np.nanmean(om_quality)
    summary["OM_EVIDENCE (B3)"][dataset_name] = np.nanmean(om_evidence)

    summary["OM_B1_B2"][dataset_name] = np.nanmean(om_b1_b2)
    summary["OM_B2_B3"][dataset_name] = np.nanmean(om_b2_b3)
    summary["OM_B1_B3"][dataset_name] = np.nanmean(om_b3_b1)

    summary["OM_MEAN"][dataset_name] = np.nanmean(om_total)

# -------- BUILD DATAFRAME IN DESIRED SHAPE -------- #

df = pd.DataFrame(summary).T  # rows = metrics, columns = datasets

# Round to 4 decimals
df = df.applymap(lambda x: round(x, 4) if isinstance(x, float) else x)

# Save CSV
df.to_csv("metric_summary_table.csv")

print("✅ Saved: metric_summary_table.csv")
print(df)

### wilcoxon and median diffs

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# -----------------------------
# CONFIG
# -----------------------------
SHOT_FILES = {
    1: "RT_Original_1shot.json",
    2: "RT_Original_2shot.json",
    4: "RT_Original_4shot.json",
}

# -----------------------------
# METRIC EXTRACTION
# -----------------------------
def extract_metrics(entry):
    """Flatten metrics needed for analysis."""
    return {
        "ROSCOE-SA": entry.get("mean_ROSCOE_SA"),
        "ROSCOE-SS": entry.get("mean_ROSCOE_SS"),
        "ROSCOE-LI": entry.get("mean_ROSCOE_LI"),
        "ROSCOE-LC": entry.get("mean_ROSCOE_LC"),
        "ROSCOE_MEAN": np.nanmean([
            entry.get("mean_ROSCOE_SA"),
            entry.get("mean_ROSCOE_SS"),
            entry.get("mean_ROSCOE_LI"),
            entry.get("mean_ROSCOE_LC"),
        ]),
        "KOTONYA_AND_TONI": entry["coherence_scores"]["mean_coherence"],
        "LLM_AS_A_JUDGE": entry.get("judge_score"),
        "RECEval": entry.get("mean_RECEval"),

        # Our Metric (OM)
        "OM_COHERENCE (B1)": entry["ourmetric"]["coherence_score"],
        "OM_QUALITY (B2)": entry["ourmetric"]["quality_score"],
        "OM_EVIDENCE (B3)": entry["ourmetric"]["evidence_score"],
        "OM_B1_B2": entry["ourmetric"]["b1_b2"],
        "OM_B2_B3": entry["ourmetric"]["b2_b3"],
        "OM_B1_B3": entry["ourmetric"]["b3_b1"],
        "OM_MEAN": entry["ourmetric"]["total_score"],
    }


# -----------------------------
# 1. LOAD SHOT-WISE FILES
# -----------------------------
shot_data = {}

for shot, fname in SHOT_FILES.items():
    path = Path(fname)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {fname}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{fname} does not contain a list")

    shot_data[shot] = data

print("✅ Loaded shot-wise files")


# -----------------------------
# 2. BUILD DATAFRAMES (ALIGNED BY CLAIM)
# -----------------------------
def build_df(entries):
    rows = []
    for e in entries:
        row = {
            "claim_id": e["claim_id"],
            "dataset": e["dataset"],
            "model": e.get("model", e["claim_id"]),
        }
        row.update(extract_metrics(e))
        rows.append(row)
    return pd.DataFrame(rows)


dfs = {shot: build_df(entries) for shot, entries in shot_data.items()}

# Align on common instances
key_cols = ["claim_id", "dataset", "model"]
df_1 = dfs[1].set_index(key_cols)
df_2 = dfs[2].set_index(key_cols)
df_4 = dfs[4].set_index(key_cols)

common_index = (
    df_1.index
    .intersection(df_2.index)
    .intersection(df_4.index)
)

df_1 = df_1.loc[common_index]
df_2 = df_2.loc[common_index]
df_4 = df_4.loc[common_index]

print(f"✅ Aligned on {len(common_index)} common instances")

# -----------------------------
# 3. WILCOXON + MEDIAN DIFF
# -----------------------------
def wilcoxon_analysis(a, b):
    diff = b - a
    diff = diff.dropna()

    if diff.nunique() <= 1:
        return np.nan, np.nan, "no-change", np.nan

    stat, p = wilcoxon(diff)
    median_diff = np.median(diff)

    # Rank-biserial correlation (effect size)
    n = len(diff)
    rbc = 1 - (2 * stat) / (n * (n + 1))

    direction = (
        "increase" if median_diff > 0 else
        "decrease" if median_diff < 0 else
        "no-change"
    )

    return median_diff, p, direction, rbc


results = []

metrics = df_1.columns

comparisons = [
    ("2-shot vs 1-shot", df_1, df_2),
    ("4-shot vs 1-shot", df_1, df_4),
    ("4-shot vs 2-shot", df_2, df_4),
]

for name, dfa, dfb in comparisons:
    for m in metrics:
        med, p, direction, effect = wilcoxon_analysis(dfa[m], dfb[m])
        results.append({
            "comparison": name,
            "metric": m,
            "median_difference": med,
            "wilcoxon_p": p,
            "effect_size_rbc": effect,
            "direction": direction,
            "significant(p<0.05)": (p < 0.05) if not np.isnan(p) else False,
        })


results_df = pd.DataFrame(results)
results_df.to_csv("Shot_Difference_Wilcoxon_Results.csv", index=False)

print("✅ Wilcoxon analysis complete")
print("📄 Results saved to Shot_Difference_Wilcoxon_Results.csv")